# 05 · Tool use with a human in the loop ⏱️ 15 minutes

A tool call is the model asking your program to do something: look up a name in the GND, query Wikidata, run a SPARQL query.
The model only *proposes*. You approve, correct or refuse; Python executes; the result goes back to the model.

Why the gate: In the 9 October 2026 probe the 8B produced a structured call for 44 of 44 English prompts, 1 of 9 German and
4 of 6 French; in the 15 October harness it scored 16 of 20, with one English SPARQL prompt whose call leaked as text
(`models/README.md`). When it did not call a tool it sometimes invented a GND number. The gate turns a confident guess into a
checked fact.

**What you'll learn**
1. Define tools as plain Python functions with docstrings
2. Let the model propose a call, approve or correct it, feed the result back
3. Chain two tools: GND identifier → Wikidata → Historical Dictionary of Switzerland
4. Recognise the failure modes and what the loop does about them

## 05.01 Configuration

In [1]:
import os, json
os.environ.setdefault("WORKSHOP_TIER", "16gb")
from workshop.config import load_settings
settings = load_settings(); MODEL = settings.model
print("Model:", MODEL, "| tool prompts in English work best with the 8B")

Model: nishad/apertus:4b | tool prompts in English work best with the 8B


## 05.02 Three tools

The docstring is the tool's description to the model; the `Args:` lines describe the parameters. `sparql` only allows SELECT
and ASK on an allow-list of endpoints and forces a LIMIT. If the network fails, each tool returns a recorded fixture and says so.

In [2]:
from workshop.tools import search_gnd, wikidata_lookup, sparql, TOOLS
print(json.loads(search_gnd("Hannah Arendt", "Person"))["results"][:2])
print(json.loads(wikidata_lookup("Hannah Arendt", "en"))["results"][:1])
print(json.loads(sparql("SELECT ?item WHERE { ?item wdt:P227 '11850391X' }", "wikidata")))

[{'image': 'https://commons.wikimedia.org/wiki/Special:FilePath/Hannah%20Arendt%20auf%20dem%201.%20Kulturkritikerkongress%2C%20Barbara%20Niggl%20Radloff%2C%20FM-2019-1-5-9-16%20%28cropped%29.jpg?width=100', 'label': 'Arendt, Hannah | 1906–1975 | Soziologin; Politologin; Philosophin; Historikerin; Schriftstellerin', 'id': 'https://d-nb.info/gnd/11850391X', 'category': 'Individualisierte Person', 'gnd': '11850391X'}, {'label': 'Arendt, Paul | 1873–1913 | Ingenieur', 'id': 'https://d-nb.info/gnd/1190671301', 'category': 'Individualisierte Person', 'gnd': '1190671301'}]


[{'id': 'Q60025', 'label': 'Hannah Arendt', 'description': 'German-American political theorist and philosopher (1906–1975)'}]


{'results': [{'item': 'http://www.wikidata.org/entity/Q60025'}], 'fixture': False, 'query': "SELECT ?item WHERE { ?item wdt:P227 '11850391X' }\nLIMIT 50"}


## 05.03 One proposed call, one decision

Ask in English. The model should propose `search_gnd`; the gate shows the call and waits for **y**, **n**, or corrected JSON arguments.

In [3]:
from workshop.tools import run_tool_loop
from workshop.display import show_markdown
messages = [{"role": "system", "content": "You are a cataloguing assistant. Use the tools to look things up; never state an identifier you did not retrieve."},
            {"role": "user", "content": "Find the GND authority record for Hannah Arendt and tell me her GND identifier."}]
out = run_tool_loop(messages, TOOLS, model=MODEL)
for m in out[len(messages):]:
    print(m["role"].upper(), "→", (m.get("content") or "")[:300], m.get("tool_calls", ""))
show_markdown(out[-1]["content"])

ASSISTANT →  [{'function': {'name': 'search_gnd', 'arguments': {'name': 'Hannah Arendt', 'type': 'Person'}}}]
TOOL → {"results": [{"image": "https://commons.wikimedia.org/wiki/Special:FilePath/Hannah%20Arendt%20auf%20dem%201.%20Kulturkritikerkongress%2C%20Barbara%20Niggl%20Radloff%2C%20FM-2019-1-5-9-16%20%28cropped%29.jpg?width=100", "label": "Arendt, Hannah | 1906–1975 | Soziologin; Politologin; Philosophin; Hist 
ASSISTANT →  The GND identifier for Hannah Arendt is `https://d-nb.info/gnd/11850391X`. 


## 05.04 Correcting the model

Names get misspelled, by people in questions and by models in tool arguments. This question spells the name *Schwartzenbach*;
the GND has it as *Schwarzenbach*, and lobid finds nobody under the misspelling. In the live session the gate shows the proposed
call and you paste the corrected JSON, `{"name": "Annemarie Schwarzenbach", "type": "Person"}` (or type **y** if the model already
fixed the spelling). The committed run cannot type at the gate, so `gate` scripts the same correction and prints the proposed and
the corrected call, so you can see it.

In [4]:
from workshop.tools import approve

def gate(name, arguments):
    """The usual approval gate. Headless (WORKSHOP_AUTO_APPROVE=1), it applies the correction you would paste."""
    approved = approve(name, arguments)
    if os.environ.get("WORKSHOP_AUTO_APPROVE") != "1" or approved is None:
        return approved
    corrected = json.loads(json.dumps(approved, ensure_ascii=False).replace("Schwartzenbach", "Schwarzenbach"))
    print(f"proposed: {name}({json.dumps(arguments, ensure_ascii=False)})")
    print(f"corrected: {name}({json.dumps(corrected, ensure_ascii=False)})")
    return corrected

if ":4b" not in MODEL:
    fix = [messages[0], {"role": "user", "content": "Look up the Swiss writer Annemarie Schwartzenbach in the GND."}]
    out = run_tool_loop(fix, TOOLS, model=MODEL, approve_fn=gate)
    for m in out[len(fix):]:
        print(m["role"].upper(), "→", (m.get("content") or "")[:300], m.get("tool_calls", ""))
    show_markdown(out[-1]["content"])
else:
    print("Skipped on the 4B: the single-tool cell above is the one to run on the 8 GB tier.")

Skipped on the 4B: the single-tool cell above is the one to run on the 8 GB tier.


Compare the model's summary with the tool message. The ASSISTANT line keeps the call as the model proposed it, misspelling
included; the TOOL line is the result of the corrected call, cut at 300 characters in the printout (the same five lobid records
are in `fixtures/search_gnd/annemarie_schwarzenbach_person.json`).

In the October 2026 run the summary attached dates and professions that the lobid result does not contain. lobid has
`Speerli, Annemarie | Künstlerin; Malerin` with no dates, and the summary gives her 1906–1999, Suzanne Schwarzenbach's dates.
lobid has `Schwarzenbach, Bettina Elisabeth Annemarie | 1979–` with no profession, and the summary calls her an agrarian
scientist, Suzanne's profession. It also calls all five people Swiss, which no lobid label says, and writes the writer's name
with the question's misspelling, *Schwartzenbach*. Nothing was marked: the gate and the `[UNVERIFIED]` check verify identifiers
only, and 118997718 did come from the tool. Prose about a person must be checked against the tool message.

## 05.05 Two tools in a row: GND → Wikidata → the Historical Dictionary of Switzerland

Wikidata links authority files: property P227 holds the GND identifier, P902 the HLS article id. One question, two tool rounds.
The loop runs one call per turn, so the second call can only be made after the first result is in.

Two-step chains are where an 8B model at this quantization gets unreliable. In the October 2026 runs it sometimes looked up the GND record and then stopped (it pasted the URL form of the id, or announced the second call without making it), and sometimes answered with an empty code block right away. Nothing was invented, which is the point of the gate. Run the cell and see what
you get; the next cell shows how you finish the chain yourself when the model stops.

**On the 8 GB tier run the single-tool cells only.** The 4B model can fall into a repetition loop that Ollama aborts with an error on the multi-step prompts (05.04 and this cell), so both cells are skipped there and the next cell makes the calls by hand.

In [5]:
chain = [messages[0], {"role": "user", "content":
    "Find the GND identifier of Annemarie Schwarzenbach with search_gnd, then use the sparql tool on wikidata to find the item with that "
    "GND id (property wdt:P227, the bare id as a string literal) and return its HLS id (property wdt:P902). "
    "Call the tools one after the other; do not describe the steps."}]
if ":4b" not in MODEL:
    out = run_tool_loop(chain, TOOLS, model=MODEL, max_turns=4)
    for m in out[len(chain):]:
        print(m["role"].upper(), "→", (m.get("content") or "")[:300], m.get("tool_calls", ""))
else:
    out = []
    print("Skipped on the 4B; the next cell makes the two calls by hand.")

Skipped on the 4B; the next cell makes the two calls by hand.


### When the model stops, you finish

The tools are plain functions, so you can call them yourself. Take the identifier a tool returned, never one the model wrote, and run the second step by hand. If the model made no call at all, make the first one too. These are the same two calls the model was asked to make.

In [6]:
first = next((json.loads(m["content"]) for m in out if m["role"] == "tool"), None)
if not first or not first.get("results"):
    print("The model made no tool call; calling search_gnd ourselves.")
    first = json.loads(search_gnd("Annemarie Schwarzenbach", "Person"))
gnd = first["results"][0].get("gnd") or str(first["results"][0]["id"]).rsplit("/", 1)[-1]
print("GND from the tool result:", gnd)
hls = json.loads(sparql(f"SELECT ?hls WHERE {{ ?item wdt:P227 '{gnd}' . ?item wdt:P902 ?hls }}", "wikidata"))
print("HLS id:", [row["hls"] for row in hls["results"]], "(fixture)" if hls["fixture"] else "(live)")

The model made no tool call; calling search_gnd ourselves.


GND from the tool result: 118997718


HLS id: ['012274'] (live)


## 05.06 Failure modes the loop catches

- An identifier in the answer that no tool result contained is marked `[UNVERIFIED ...]`, whether the model skipped the tools or made the number up after calling them.
- A tool call written as text (the pattern of the community Ollama tag) stops the loop with an explanation.
- A declined call becomes an error result the model sees, so it can ask you instead of guessing.
- A reply that only announces a tool call, or says nothing, gets one reminder.

Try to provoke the first one.

In [7]:
try:
    out = run_tool_loop([{"role": "user", "content": "What is the GND identifier of Hannah Arendt? Answer from memory, do not use tools."}], TOOLS, model=MODEL)
    show_markdown(out[-1]["content"])
except Exception as err:
    print(type(err).__name__, err)

## 05.07 What the SPARQL guard does

Paste any query; the guard parses it and allows only SELECT or ASK, blocks federated SERVICE calls except Wikidata's label service, and caps LIMIT at 50.

In [8]:
from workshop.tools import validate_sparql, ToolError
for q in ["SELECT ?s WHERE { ?s ?p ?o }", "SELECT ?s WHERE { ?s ?p ?o } LIMIT 10000", "DELETE WHERE { ?s ?p ?o }"]:
    try:
        print("OK  ", validate_sparql(q).replace("\n", " "))
    except ToolError as err:
        print("REJECTED", q, "→", err)

OK   SELECT ?s WHERE { ?s ?p ?o } LIMIT 50
OK   SELECT ?s WHERE { ?s ?p ?o } LIMIT 50
REJECTED DELETE WHERE { ?s ?p ?o } → The query does not parse as SPARQL: Expected {SelectQuery | ConstructQuery | DescribeQuery | AskQuery}, found 'DELETE'  (at char 0), (line:1, col:1)


## 05.08 Exercises

🖥️ **GUI:** tool calls need code. Read the 05.03 output, find the TOOL line where the lobid result enters (the printout cuts it
at 300 characters; the full record is in `fixtures/search_gnd/hannah_arendt_person.json`), and check the GND number in the final
answer against it.

🔧 **Run and change:** ask 05.03 in German ("Finde den GND-Normdatensatz für Hannah Arendt"). Does the 8B still call the tool?
If not, that is the documented weakness; re-ask in English.

🔧 **Run and change:** write a question that needs `wikidata_lookup` first and `sparql` second (for example: which works by Goethe
does Wikidata list as novels, in German labels).

🚀 **Stretch:** fetch the HLS article found in 05.05 (CC BY-SA) with `requests`, chunk it with `workshop.rag.chunk_text`, index it and
ask a question about Schwarzenbach's travels. That is the full chain: authority file → knowledge graph → text → grounded answer.

🚀 **Stretch (8 GB tier):** run 05.03 with `nishad/apertus:4b`. It needs the wrapper tag and only simple tools; compare behaviour.

## ✅ Checkpoint

- Tools are functions; the model proposes calls; you decide; results flow back
- Identifiers that were not looked up are marked, and text-shaped calls stop the loop
- Two tools chained a GND identifier to a Wikidata item and an HLS article: by the model when it manages, by you when it stops
- You know when to re-ask in English and when to refuse a call

You are done with Part 2. Keep `runs/log.jsonl`: it is the record of what model produced which result.